# Trilateration Lab — Analysis

Monte Carlo simulation comparing 5 measurement strategies across 6 noise levels.
A box is hidden in the continental US; each week you pick a location and learn the
distance. Goal: localize within 5 miles.

Charts are interactive — hover for values, click legend items to toggle series.
HTML versions are saved to `data/` for sharing.

> **Note:** Info-Gain numbers use 100 trials per mode (25 for Gaussian σ=25 due to compute cost).
> All other strategies use 500 trials.

In [1]:
import sys
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

STRATEGIES = ["fixed", "random", "max_separation", "centroid", "info_gain"]
MODES = [
    "EXACT",
    "ROUND_10_MILES",
    "ROUND_25_MILES",
    "ROUND_100_MILES",
    "NOISY_GAUSSIAN_5",
    "NOISY_GAUSSIAN_25",
]
STRATEGY_LABELS = {
    "fixed": "Fixed",
    "random": "Random",
    "max_separation": "Max-Sep",
    "centroid": "Centroid",
    "info_gain": "Info-Gain*",
}
MODE_LABELS = {
    "EXACT": "Exact",
    "ROUND_10_MILES": "Round 10mi",
    "ROUND_25_MILES": "Round 25mi",
    "ROUND_100_MILES": "Round 100mi",
    "NOISY_GAUSSIAN_5": "Gaussian σ=5",
    "NOISY_GAUSSIAN_25": "Gaussian σ=25",
}
COLORS = px.colors.qualitative.Plotly

with open(ROOT / "data" / "results.json") as f:
    raw = json.load(f)

rows = []
for key, stats in raw.items():
    name, mode = key.split("|")
    if name in STRATEGIES and mode in MODES:
        rows.append({"strategy": name, "mode": mode,
                     "strategy_label": STRATEGY_LABELS[name],
                     "mode_label": MODE_LABELS[mode], **stats})
df = pd.DataFrame(rows)
print(f"Loaded {len(df)} configs")

Loaded 30 configs


## Failure rate heatmap

% of trials that hit the 52-week timeout without localizing within 5 miles.

In [2]:
row_order = [STRATEGY_LABELS[s] for s in STRATEGIES]
col_order = [MODE_LABELS[m] for m in MODES]

pivot = df.pivot(index="strategy_label", columns="mode_label", values="failure_rate")
pivot = pivot.loc[row_order, col_order]

hover = df.pivot(index="strategy_label", columns="mode_label", values="mean")
hover = hover.loc[row_order, col_order]

text = [[f"{v:.0%}" for v in row] for row in pivot.values]
customdata = hover.values

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=col_order,
    y=row_order,
    text=text,
    texttemplate="%{text}",
    textfont={"size": 13},
    customdata=customdata,
    hovertemplate="<b>%{y} / %{x}</b><br>Failure rate: %{z:.1%}<br>Mean weeks: %{customdata:.1f}<extra></extra>",
    colorscale=[
        [0.0, "#1a7f3c"],
        [0.2, "#7dbb6f"],
        [0.4, "#d4e88a"],
        [0.6, "#f4c04a"],
        [0.8, "#e8703a"],
        [1.0, "#c0392b"],
    ],
    zmin=0, zmax=1,
    colorbar=dict(title="Failure rate", tickformat=".0%"),
))
fig.update_layout(
    title=dict(text="Failure rate by strategy and measurement mode", font_size=16),
    xaxis=dict(title="Measurement mode", side="bottom"),
    yaxis=dict(title="Strategy", autorange="reversed"),
    height=340,
    margin=dict(l=100, r=100, t=60, b=60),
    font=dict(family="Inter, system-ui, sans-serif"),
)
fig.write_html(ROOT / "data" / "failure_heatmap.html")
fig.show()

## Mean weeks to localize

Includes 2-week ground search penalty. Timed-out trials count as 52 weeks,
so this captures both speed and reliability in one number.

In [3]:
pivot_m = df.pivot(index="strategy_label", columns="mode_label", values="mean")
pivot_m = pivot_m.loc[row_order, col_order]
pivot_f = df.pivot(index="strategy_label", columns="mode_label", values="failure_rate")
pivot_f = pivot_f.loc[row_order, col_order]

text_m = [[f"{v:.0f}w" for v in row] for row in pivot_m.values]

fig = go.Figure(go.Heatmap(
    z=pivot_m.values,
    x=col_order,
    y=row_order,
    text=text_m,
    texttemplate="%{text}",
    textfont={"size": 13},
    customdata=pivot_f.values,
    hovertemplate="<b>%{y} / %{x}</b><br>Mean weeks: %{z:.1f}<br>Failure rate: %{customdata:.1%}<extra></extra>",
    colorscale=[
        [0.0, "#1a7f3c"],
        [0.2, "#7dbb6f"],
        [0.5, "#f4c04a"],
        [1.0, "#c0392b"],
    ],
    zmin=5, zmax=52,
    colorbar=dict(title="Mean weeks"),
))
fig.update_layout(
    title=dict(text="Mean weeks to localize (lower is better)", font_size=16),
    xaxis=dict(title="Measurement mode", side="bottom"),
    yaxis=dict(title="Strategy", autorange="reversed"),
    height=340,
    margin=dict(l=100, r=100, t=60, b=60),
    font=dict(family="Inter, system-ui, sans-serif"),
)
fig.write_html(ROOT / "data" / "mean_weeks_heatmap.html")
fig.show()

## Failure rate vs noise level

How each strategy degrades as precision gets worse.
Click legend items to isolate a strategy. When lines converge, noise dominates and strategy choice stops mattering.

In [4]:
fig = go.Figure()
mode_labels = [MODE_LABELS[m] for m in MODES]

dash_styles = ["solid", "solid", "solid", "solid", "dash"]

for (strategy, label), color, dash in zip(STRATEGY_LABELS.items(), COLORS, dash_styles):
    subset = df[df["strategy"] == strategy].set_index("mode")
    y = [subset.loc[m, "failure_rate"] if m in subset.index else None for m in MODES]
    mean_w = [subset.loc[m, "mean"] if m in subset.index else None for m in MODES]
    p90 = [subset.loc[m, "p90"] if m in subset.index else None for m in MODES]

    fig.add_trace(go.Scatter(
        x=mode_labels,
        y=y,
        name=label,
        mode="lines+markers",
        line=dict(color=color, width=2.5, dash=dash),
        marker=dict(size=8),
        customdata=list(zip(mean_w, p90)),
        hovertemplate=(
            "<b>" + label + " / %{x}</b><br>"
            "Failure: %{y:.1%}<br>"
            "Mean: %{customdata[0]:.1f}w<br>"
            "P90: %{customdata[1]:.0f}w"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    title=dict(text="Failure rate across measurement modes", font_size=16),
    xaxis=dict(title="Measurement mode (increasing noise →)"),
    yaxis=dict(title="Failure rate", tickformat=".0%", range=[-0.02, 1.05]),
    legend=dict(title="Strategy", bgcolor="rgba(255,255,255,0.9)",
                bordercolor="#ddd", borderwidth=1),
    hovermode="x unified",
    height=460,
    margin=dict(l=60, r=40, t=60, b=60),
    font=dict(family="Inter, system-ui, sans-serif"),
    plot_bgcolor="#fafafa",
    paper_bgcolor="white",
)
fig.add_annotation(
    text="* Info-Gain: 100 trials/mode (25 for Gaussian σ=25), dashed line",
    xref="paper", yref="paper", x=0, y=-0.15,
    showarrow=False, font=dict(size=10, color="gray"), align="left",
)
fig.write_html(ROOT / "data" / "failure_by_mode.html")
fig.show()

## Does strategy choice matter?

Range of failure rates across strategies for each mode.
Large bar = strategy choice is the deciding factor. Small bar = noise level dominates.

In [5]:
spread = (
    df.groupby("mode")["failure_rate"]
    .agg(spread=lambda x: x.max() - x.min(),
         best=lambda x: x.min(),
         worst=lambda x: x.max())
    .reindex(MODES)
    .reset_index()
)
spread["mode_label"] = spread["mode"].map(MODE_LABELS)

# Best and worst strategy per mode for hover
best_strat, worst_strat = [], []
for mode in MODES:
    sub = df[df["mode"] == mode]
    best_strat.append(STRATEGY_LABELS[sub.loc[sub["failure_rate"].idxmin(), "strategy"]])
    worst_strat.append(STRATEGY_LABELS[sub.loc[sub["failure_rate"].idxmax(), "strategy"]])

fig = go.Figure(go.Bar(
    x=spread["mode_label"],
    y=spread["spread"],
    marker=dict(
        color=spread["spread"],
        colorscale=[[0, "#a8d5b5"], [0.4, "#f4c04a"], [1, "#c0392b"]],
        showscale=True,
        colorbar=dict(title="Spread", tickformat=".0%"),
    ),
    customdata=list(zip(spread["best"], spread["worst"], best_strat, worst_strat)),
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Strategy spread: %{y:.0%}<br>"
        "Best: %{customdata[2]} (%{customdata[0]:.0%} fail)<br>"
        "Worst: %{customdata[3]} (%{customdata[1]:.0%} fail)"
        "<extra></extra>"
    ),
    text=[f"{v:.0%}" for v in spread["spread"]],
    textposition="outside",
))
fig.update_layout(
    title=dict(text="How much does strategy choice matter?", font_size=16),
    xaxis=dict(title="Measurement mode"),
    yaxis=dict(title="Max − min failure rate across strategies",
               tickformat=".0%", range=[0, 1.1]),
    height=420,
    margin=dict(l=60, r=60, t=60, b=60),
    font=dict(family="Inter, system-ui, sans-serif"),
    plot_bgcolor="#fafafa",
    paper_bgcolor="white",
    showlegend=False,
)
fig.write_html(ROOT / "data" / "strategy_spread.html")
fig.show()

## P90 and median — distribution of weeks to localize

Median shows the typical case. P90 shows the tail — a P90 of 52 means at least
10% of trials timed out. Toggle between metrics with the buttons.

In [6]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Median weeks", "P90 weeks"),
    shared_yaxes=False,
)

for metric, col, title in [("median", 1, "Median"), ("p90", 2, "P90")]:
    pivot = df.pivot(index="strategy_label", columns="mode_label", values=metric)
    pivot = pivot.loc[row_order, col_order]
    text_vals = [[f"{v:.0f}" for v in row] for row in pivot.values]

    fig.add_trace(go.Heatmap(
        z=pivot.values,
        x=col_order,
        y=row_order,
        text=text_vals,
        texttemplate="%{text}",
        textfont={"size": 11},
        hovertemplate=f"<b>%{{y}} / %{{x}}</b><br>{title}: %{{z:.0f}} weeks<extra></extra>",
        colorscale=[
            [0.0, "#1a7f3c"],
            [0.3, "#7dbb6f"],
            [0.6, "#f4c04a"],
            [1.0, "#c0392b"],
        ],
        zmin=5, zmax=52,
        showscale=(col == 2),
        colorbar=dict(title="Weeks", x=1.02) if col == 2 else None,
    ), row=1, col=col)

fig.update_layout(
    title=dict(text="Distribution of weeks to localize", font_size=16),
    height=340,
    margin=dict(l=100, r=100, t=80, b=40),
    font=dict(family="Inter, system-ui, sans-serif"),
)
for col in [1, 2]:
    fig.update_yaxes(autorange="reversed", row=1, col=col)
    fig.update_xaxes(tickangle=30, row=1, col=col)

fig.write_html(ROOT / "data" / "distribution.html")
fig.show()